In [2]:
import pandas as pd
from sklearn.ensemble import IsolationForest


In [7]:
# Load the CSV file into a DataFrame
data = pd.read_csv('../data/data_uncorr.csv')

# Display the first few rows of the DataFrame
data.head()

,Unnamed: 0,name,market,funding_total_usd,status,country_code,state_code,region,city,funding_rounds,...,round_B,round_C,round_D,round_E,round_F,round_G,international,european_or_international,time_to_first_funding,status_encoded
0,0,#waywire,News,1750000.0,acquired,USA,NY,New York City,New York,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,international,0.079452,0
1,1,'Rock' Your Paper,Publishing,40000.0,operating,EST,other,Tallinn,Tallinn,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,european,-0.213699,2
2,2,(In)Touch Network,Electronics,1500000.0,operating,GBR,other,London,London,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,european,0.000000,2
3,3,-R- Ranch and Mine,Tourism,60000.0,operating,USA,TX,Dallas,Fort Worth,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,international,0.624658,2
4,4,0-6.com,Curated Web,2000000.0,operating,other,other,other,other,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,international,1.213699,2


# One hot encoding

#### Binary Representation of Funding Rounds

In [8]:
data['had_round_A'] = [0 if x==0 else 1 for x in data['round_A']]
data['had_round_B'] = [0 if x==0 else 1 for x in data['round_B']]
data['had_round_C'] = [0 if x==0 else 1 for x in data['round_C']]
data['had_round_D'] = [0 if x==0 else 1 for x in data['round_D']]
data['had_round_E'] = [0 if x==0 else 1 for x in data['round_E']]
data['had_round_F'] = [0 if x==0 else 1 for x in data['round_F']]
data['had_round_G'] = [0 if x==0 else 1 for x in data['round_G']]
data['had_venture'] = [0 if x==0 else 1 for x in data['venture']]
data['had_seed'] = [0 if x==0 else 1 for x in data['seed']]
data['had_eq_crowdfunding'] = [0 if x==0 else 1 for x in data['equity_crowdfunding']]
data['had_pd_crowdfunding'] = [0 if x==0 else 1 for x in data['product_crowdfunding']]
data['had_angel'] = [0 if x==0 else 1 for x in data['angel']]
data['had_grant'] = [0 if x==0 else 1 for x in data['grant']]
data['had_pe'] = [0 if x==0 else 1 for x in data['private_equity']]
data['had_convert'] = [0 if x==0 else 1 for x in data['convertible_note']]

In [9]:
data.columns

Index(['Unnamed: 0', 'name', 'market', 'funding_total_usd', 'status',
       'country_code', 'state_code', 'region', 'city', 'funding_rounds',
       'founded_month', 'founded_year', 'seed', 'venture',
       'equity_crowdfunding', 'undisclosed', 'convertible_note',
       'debt_financing', 'angel', 'grant', 'private_equity',
       'product_crowdfunding', 'round_A', 'round_B', 'round_C', 'round_D',
       'round_E', 'round_F', 'round_G', 'international',
       'european_or_international', 'time_to_first_funding', 'status_encoded',
       'had_round_A', 'had_round_B', 'had_round_C', 'had_round_D',
       'had_round_E', 'had_round_F', 'had_round_G', 'had_venture', 'had_seed',
       'had_eq_crowdfunding', 'had_pd_crowdfunding', 'had_angel', 'had_grant',
       'had_pe', 'had_convert'],
      dtype='object')

### Market Groups

In [10]:
# Define the mapping of markets to market groups
market_groups_mapping = {
    "Software & Tecnologie Digitali": [
        "Software", "Mobile", "Enterprise Software", "SaaS", "Cloud Computing",
        "Analytics", "AI & Machine Learning", "Data Security", "IT & Cybersecurity"
    ],
    "Salute, Benessere & Biotech": [
        "Health Care", "Health and Wellness", "Medical", "Pharmaceuticals",
        "Medical Devices", "Mobile Health", "Bioinformatics", "Healthcare Services"
    ],
    "E-Commerce & Retail": [
        "E-Commerce", "Online Shopping", "Mobile Commerce", "Retail",
        "Marketplaces", "Consumer Goods", "Fashion", "Mobile Advertising"
    ],
    "Media, Comunicazione & Intrattenimento": [
        "Social Media", "Social Network Media", "Social Media Marketing",
        "Entertainment", "Music", "Video Streaming", "Social Games", "Online Video Advertising"
    ],
    "Finanza & Business": [
        "Finance", "Financial Services", "Payments", "Venture Capital",
        "Crowdfunding", "Personal Finance", "Investment Management", "Business Analytics"
    ],
    "Educazione & Formazione": [
        "Education", "K-12 Education", "Career Management", "Training",
        "Language Learning", "Tutoring"
    ],
    "Tecnologie Innovative & Energia": [
        "Clean Technology", "Renewable Energies", "Solar", "Energy Efficiency",
        "Energy IT", "Smart Grid"
    ],
    "Automotive & Trasporti": [
        "Automotive", "Transportation", "Cars", "Fleet Management", "Taxis"
    ],
    "Real Estate & Immobili": [
        "Real Estate", "Property Management", "Commercial Real Estate", "Residential Solar"
    ],
    "Diversi & Altri Settori": [
        "Startups", "Consulting", "Nonprofits", "Sports", "Events",
        "Crowdsourcing", "Art", "Charity"
    ]
}

# Create a reverse mapping for easier lookup
reverse_mapping = {market: group for group, markets in market_groups_mapping.items() for market in markets}

# Add the "market groups" column to the dataframe
data["market groups"] = data["market"].map(reverse_mapping).fillna("Unknown")

In [11]:
data["market groups"].value_counts()

market groups
Unknown                                   18810
Software & Tecnologie Digitali             6366
E-Commerce & Retail                        2056
Salute, Benessere & Biotech                1656
Media, Comunicazione & Intrattenimento     1401
Diversi & Altri Settori                     877
Finanza & Business                          844
Tecnologie Innovative & Energia             716
Educazione & Formazione                     686
Real Estate & Immobili                      340
Automotive & Trasporti                      288
Name: count, dtype: int64

In [12]:
# Perform one-hot encoding on the 'market groups' column
market_groups_encoded = pd.get_dummies(data['market groups'], prefix='market_group')

# Concatenate the encoded columns back to the original dataframe
data = pd.concat([data, market_groups_encoded], axis=1)

# Convert True/False to 0/1 for all boolean columns
market_groups_encoded = market_groups_encoded.astype(int)

# Aggiorna il dataframe originale
data.update(market_groups_encoded)

/tmp/ipykernel_124049/1548527883.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 0]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  data.update(market_groups_encoded)
/tmp/ipykernel_124049/1548527883.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 0]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  data.update(market_groups_encoded)
/tmp/ipykernel_124049/1548527883.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 0]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  data.update(market_groups_encoded)
/tmp/ipykernel_124049/1548527883.py:11: FutureWarning: Setting an item of incompatible dtype is depr

In [13]:
data.columns

Index(['Unnamed: 0', 'name', 'market', 'funding_total_usd', 'status',
       'country_code', 'state_code', 'region', 'city', 'funding_rounds',
       'founded_month', 'founded_year', 'seed', 'venture',
       'equity_crowdfunding', 'undisclosed', 'convertible_note',
       'debt_financing', 'angel', 'grant', 'private_equity',
       'product_crowdfunding', 'round_A', 'round_B', 'round_C', 'round_D',
       'round_E', 'round_F', 'round_G', 'international',
       'european_or_international', 'time_to_first_funding', 'status_encoded',
       'had_round_A', 'had_round_B', 'had_round_C', 'had_round_D',
       'had_round_E', 'had_round_F', 'had_round_G', 'had_venture', 'had_seed',
       'had_eq_crowdfunding', 'had_pd_crowdfunding', 'had_angel', 'had_grant',
       'had_pe', 'had_convert', 'market groups',
       'market_group_Automotive & Trasporti',
       'market_group_Diversi & Altri Settori',
       'market_group_E-Commerce & Retail',
       'market_group_Educazione & Formazione',
    

In [15]:
# Initialize the Isolation Forest model
iso_forest = IsolationForest(random_state=42, contamination=0.05)

# Fit the model and predict anomalies
isolation_forest_pred = iso_forest.fit_predict(data[numerical_columns])

# Outliers are marked as -1
outliers_isolation_forest = data[numerical_columns][isolation_forest_pred == -1]

# Calculate the percentage of outliers
outlier_percentage = (len(outliers_isolation_forest) / len(data)) * 100
print(f"Percentage of outliers: {outlier_percentage:.2f}%")

Percentage of outliers: 5.00%


In [16]:
# Filter out the outliers
data_no_outliers = data[isolation_forest_pred != -1]

# Display the shape of the new dataset
print(f"Shape of dataset after removing outliers: {data_no_outliers.shape}")

Shape of dataset after removing outliers: (32338, 61)


In [17]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Standardize the numerical columns
data_no_outliers[numerical_columns] = scaler.fit_transform(data_no_outliers[numerical_columns])

# Display the first few rows of the standardized data
data_no_outliers.head()

/tmp/ipykernel_124049/2244712812.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_no_outliers[numerical_columns] = scaler.fit_transform(data_no_outliers[numerical_columns])


,Unnamed: 0,name,market,funding_total_usd,status,country_code,state_code,region,city,funding_rounds,...,market_group_E-Commerce & Retail,market_group_Educazione & Formazione,market_group_Finanza & Business,"market_group_Media, Comunicazione & Intrattenimento",market_group_Real Estate & Immobili,"market_group_Salute, Benessere & Biotech",market_group_Software & Tecnologie Digitali,market_group_Tecnologie Innovative & Energia,market_group_Unknown,anomaly
0,-1.737992,#waywire,News,-0.157341,acquired,USA,NY,New York City,New York,-0.586582,...,-0.254147,-0.144454,-0.157214,-0.207429,-0.101538,-0.219776,-0.479147,-0.136323,0.890321,0
1,-1.737890,'Rock' Your Paper,Publishing,-0.223259,operating,EST,other,Tallinn,Tallinn,-0.586582,...,-0.254147,-0.144454,-0.157214,-0.207429,-0.101538,-0.219776,-0.479147,-0.136323,0.890321,0
2,-1.737788,(In)Touch Network,Electronics,-0.166978,operating,GBR,other,London,London,-0.586582,...,-0.254147,-0.144454,-0.157214,-0.207429,-0.101538,-0.219776,-0.479147,-0.136323,0.890321,0
3,-1.737686,-R- Ranch and Mine,Tourism,-0.222488,operating,USA,TX,Dallas,Fort Worth,0.318725,...,-0.254147,-0.144454,-0.157214,-0.207429,-0.101538,-0.219776,-0.479147,-0.136323,0.890321,0
4,-1.737584,0-6.com,Curated Web,-0.147704,operating,other,other,other,other,-0.586582,...,-0.254147,-0.144454,-0.157214,-0.207429,-0.101538,-0.219776,-0.479147,-0.136323,0.890321,0


In [ ]:
# Uncomment to Standardize between 0 and 1, but comment the StandardScaler part

# from sklearn.preprocessing import MinMaxScaler

# # Initialize the MinMaxScaler
# min_max_scaler = MinMaxScaler()

# # Apply MinMaxScaler to the numerical columns
# data_no_outliers[numerical_columns] = min_max_scaler.fit_transform(data_no_outliers[numerical_columns])

# # Display the first few rows of the standardized data
# data_no_outliers.head()